# JRA-3Q 海面更正気圧（気圧配置）一括ダウンロード（Colab版）

手元のPC/ネットワークから `github.com` や GDEX（データ配布元）へのHTTPS通信がブロックされる環境向けに、Google Colab上でダウンロードするノートブックです。Colab（Googleのサーバー）から直接ダウンロードするので、手元の回線・セキュリティソフトの制限は関係なくなります。

## 保存先・セッション切れ対策について
**Googleドライブは使いません**（容量を圧迫しないため）。代わりに:
1. Colab上の一時ディスク（無料・数十GBの空きがあります）にダウンロードする
2. ある程度たまったら **ZIPにまとめてブラウザ経由でお使いのPCに直接ダウンロード**（ブラウザの通信は制限されていないので、これは問題なく動くはずです）
3. **⑤のセルは1回の実行につき1バッチだけ処理します**（既定10ヶ月分≈850MB）。複数バッチを1回のセル実行でまとめて処理すると、ブラウザへのダウンロードが詰まって止まってしまう不具合が起きやすいためです。1バッチ終わるごとに、もう一度同じセルを実行してください（`START_BATCH` が自動で1つ進みます）。

## 途中で切れた・止まった場合
1. Colabのページを**再読み込み**（F5）する（止まったままのダウンロード表示を消すため）
2. ランタイムを再接続
3. PCのダウンロードフォルダを確認し、`jra3q_pressure_batchNN.zip` で**実際にサイズが入っている**（0バイトや「削除済み」ではない）ものの最大のNNを確認する
4. ①③④を実行し直し、④の `START_BATCH` をその NN+1 に書き換えてから、⑤を1回ずつ実行していく

## 使い方
上から順にセルを実行してください（Shift+Enter）。⑤だけは、必要バッチ数ぶん何度も実行してください。

## ① リポジトリを取得（初回はクローン、2回目以降は最新化のみ）

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/typhoon-dataset-improvements-hn814c'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} https://github.com/awg-yk/typhoon-wind-rainfall {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

## ② 対象月・バッチ数を確認

`download_jra3q_pressure.py` の中の関数をそのまま使い、必要な月の一覧を取得します（本体スクリプトはCLIとしても単独で使えますが、ここではバッチ分割のため関数を直接呼び出します）。

In [ ]:
import sys
sys.path.insert(0, f'{REPO_DIR}/scripts')
import download_jra3q_pressure as jra

INCLUDE_SURFACE_PRESSURE = False  # True にすると地上気圧(pres-sfc)も追加でダウンロード

VARIABLES = list(jra.VARIABLES)
if INCLUDE_SURFACE_PRESSURE:
    VARIABLES.append(jra.SURFACE_PRESSURE)

MONTHS = jra.needed_year_months()
FILES_PER_BATCH = 10  # 1バッチ ≈ 10ファイル ≈ 850MB (prmsl-msl単体の場合)

jobs = [(y, m, var_code, var_name) for (y, m) in MONTHS for var_code, var_name in VARIABLES]
batches = [jobs[i:i + FILES_PER_BATCH] for i in range(0, len(jobs), FILES_PER_BATCH)]

print(f'{len(MONTHS)} months x {len(VARIABLES)} variable(s) = {len(jobs)} files')
print(f'{len(batches)} batches of up to {FILES_PER_BATCH} files (~{FILES_PER_BATCH*85/1000:.2f} GB each)')

## ③ ローカル作業フォルダの準備

In [ ]:
import pathlib

LOCAL_DIR = pathlib.Path('/content/jra3q_pressure')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

## ④ 開始バッチ番号を設定

初回だけ実行してください（`START_BATCH = 0`）。⑤を実行するたびに自動で1つずつ増えるので、それ以降はこのセルを再実行しないでください。セッションが切れて再開する場合だけ、このセルを実行し直して `START_BATCH` を「PCに確実に保存できている最大のバッチ番号 + 1」に書き換えてください。

In [ ]:
START_BATCH = 0

## ⑤ 1バッチぶんダウンロード＆ZIP化＆PCへのダウンロード（本体・繰り返し実行）

**このセルを、`全バッチ完了しました` と表示されるまで何度も実行してください。** 1回の実行で: 現在のバッチをダウンロード → ZIPにまとめる → ブラウザ経由でPCにダウンロード → Colab側の一時ファイルを削除 → `START_BATCH` を1つ進める、をやります。

ブラウザが「複数ファイルのダウンロード」を確認するポップアップを出すことがあります。その場合は許可してください。1回のセル実行につき1バッチだけなので、途中で止まった場合に失われるのも最大1バッチ分だけです。

In [ ]:
import zipfile
from google.colab import files as colab_files

if START_BATCH >= len(batches):
    print('全バッチ完了しました。')
else:
    bi = START_BATCH
    batch = batches[bi]
    print(f'=== batch {bi+1}/{len(batches)} ({len(batch)} files) ===')

    saved_paths = []
    for y, m, var_code, var_name in batch:
        out_path = None
        for dataset in jra.DATASETS:
            url = jra.build_url(dataset, var_code, var_name, y, m)
            filename = url.rsplit('/', 1)[-1]
            candidate = LOCAL_DIR / filename
            if candidate.exists():
                out_path = candidate
                break
            if jra.download_one(url, candidate):
                out_path = candidate
                break
        if out_path:
            saved_paths.append(out_path)
            print(f'  {y:04d}-{m:02d} {var_name}: ok')
        else:
            print(f'  {y:04d}-{m:02d} {var_name}: FAILED (not found in any dataset)')

    if not saved_paths:
        print('  no files downloaded for this batch, skipping zip/download')
    else:
        zip_path = f'/content/jra3q_pressure_batch{bi:02d}.zip'
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as zf:
            for p in saved_paths:
                zf.write(p, arcname=p.name)
        print(f'  zipped -> {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB), triggering browser download...')
        colab_files.download(zip_path)

        for p in saved_paths:
            p.unlink()
        os.remove(zip_path)
        print(f'  batch {bi+1} done, local temp files cleared')

    START_BATCH += 1
    remaining = len(batches) - START_BATCH
    if remaining > 0:
        print(f'  次: このセルをもう一度実行してください（残り {remaining} バッチ）')
    else:
        print('  全バッチ完了しました。')

## ⑥ （任意）無操作切断を遅らせる

Colabは無操作が続くと自動切断されることがあります。⑤を繰り返す前にこのセルを流しておくと、ブラウザのタブを開いたままにしている間は接続維持の合図を送り続けます（非公式の小技のため過信せず、切れたら④→⑤の順で再実行してください）。

In [ ]:
from IPython.display import Javascript, display

display(Javascript('''
function KeepAlive(){
  console.log("keep-alive ping");
  document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(KeepAlive, 60000);
'''))

## PCで受け取った後

ダウンロードフォルダに溜まった `jra3q_pressure_batchNN.zip` を、リポジトリの `data/raw_jra3q/` フォルダなど好きな場所に展開してまとめてください。`data/raw_jra3q/` は `.gitignore` 済みなので、そのまま置いてもリポジトリには影響しません。